# Bunion angle model: GPU training

Runs the same `bunion/` code on a Colab GPU with a bigger model and sharper images.

1. **Runtime > Change runtime type > T4 GPU** (or better), then **Runtime > Run all**.
2. The clone cell asks for a GitHub token (the repo is private). Make one at github.com > Settings > Developer settings > Personal access tokens, with read access to `flatfoot-screener`.
3. At the end you get two `.onnx` files and the test scores to download.

The HVAngleEst data (CC BY-NC-ND 4.0) is downloaded fresh each time and never saved to the repo.

In [ ]:
!nvidia-smi -L

In [ ]:
import getpass, os
token = getpass.getpass('GitHub token: ')
!git clone -q https://{token}@github.com/swastikjena08-ao/flatfoot-screener.git
del token
%cd flatfoot-screener/ai_models/bunion

In [ ]:
# Dataset: about 5,600 files, a few minutes
!python download_data.py --out /content/hvangleest

## Settings
Bigger backbone and higher resolution than the CPU run (ResNet-18, 256x512 / 256x384, 30 / 25 rounds).

In [ ]:
ARCH = 'resnet34'
P1_W, P1_H, P1_EPOCHS = 320, 640, 60   # whole-foot pass
P2_W, P2_H, P2_EPOCHS = 320, 480, 60   # zoomed forefoot pass
W = '/content/work'
os.makedirs(W, exist_ok=True)

## Pass 1: whole foot

In [ ]:
!python prep.py --data /content/hvangleest --out {W}/cache.npz --width {P1_W} --height {P1_H}
!python train.py --cache {W}/cache.npz --weights imagenet --arch {ARCH} --out {W}/pass1 --epochs {P1_EPOCHS}
!python evaluate.py --cache {W}/cache.npz --ckpt {W}/pass1/best.pt --arch {ARCH} --out {W}/pass1

## Pass 2: zoomed forefoot

In [ ]:
!python zoom_prep.py --data /content/hvangleest --cache {W}/cache.npz --ckpt {W}/pass1/best.pt --arch {ARCH} --out {W}/zoom_cache.npz --width {P2_W} --height {P2_H}
!python train.py --cache {W}/zoom_cache.npz --weights imagenet --arch {ARCH} --out {W}/pass2 --epochs {P2_EPOCHS}
!python evaluate.py --cache {W}/zoom_cache.npz --ckpt {W}/pass2/best.pt --arch {ARCH} --out {W}/pass2

## Results
Paper (HVAngleEst baseline): HVA 94.4% and IMA 79.4% within 3°.

In [ ]:
import json
for p in ['pass1', 'pass2']:
    s = json.load(open(f'{W}/{p}/test_scores.json'))
    for k in ['test_official', 'test_clean']:
        r = s[k]
        print(f"{p} {k:14s} HVA {r['hva_within3']:.1%} (MAE {r['hva_mae']:.2f}°)  IMA {r['ima_within3']:.1%} (MAE {r['ima_mae']:.2f}°)")

In [ ]:
# Download the models and scores
!cd {W} && zip -q -j bunion_results.zip pass1/bunion_points.onnx pass1/test_scores.json pass1/log.csv && \
  mkdir -p p2 && cp pass2/bunion_points.onnx p2/bunion_zoom_points.onnx && \
  zip -q -j bunion_results.zip p2/bunion_zoom_points.onnx && cp pass2/test_scores.json p2/zoom_test_scores.json && cp pass2/log.csv p2/zoom_log.csv && \
  zip -q -j bunion_results.zip p2/zoom_test_scores.json p2/zoom_log.csv
from google.colab import files
files.download(f'{W}/bunion_results.zip')